In [ ]:
# AI-Based Water Contamination Detection System
**Using Machine Learning to predict water potability from physicochemical features.**

**Author:** Rabeya Zaman — Microbiology, Notre Dame University Bangladesh  
**Course:** MBO 104 — Organic Chemistry & Environmental Microbiology  
**Year:** 2026

---

## Why this project matters
Waterborne pathogens (cholera, *E. coli* O157:H7, *Salmonella*, *Giardia*, hepatitis A virus) kill hundreds of thousands of people every year — most of them in low-income countries. Microbiological culture takes 24–72 hours. **Machine learning on cheap physicochemical sensors (pH, turbidity, conductivity, chloramines, sulfate, TDS)** can give a real-time first-pass risk score so that labs prioritize which samples to culture first.

## What this notebook does
1. Loads the Kaggle Water Potability dataset (or a synthetic version with the same schema).
2. Performs EDA — class balance, distributions by potability, correlations.
3. Cleans the data (median imputation) and standardizes the features.
4. Trains **Logistic Regression**, **Decision Tree**, and **Random Forest**.
5. Evaluates all three with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.
6. Visualizes feature importance and explains it microbiologically.

## 1. Setup

In [ ]:
# If running in Colab/Kaggle, install deps:
# !pip install -q pandas numpy matplotlib scikit-learn joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
)

RANDOM_STATE = 42
plt.rcParams['figure.dpi'] = 110

## 2. Load the dataset

**On Kaggle:** attach `adityakadiwal/water-potability` and set:
```python
DATA_PATH = '/kaggle/input/water-potability/water_potability.csv'
```

**Locally:** put `water_potability.csv` in `data/` (the cell below also falls back to a synthetic dataset).

In [ ]:
import sys
sys.path.append('..')  # so we can import the src/ package from the notebook

from src.data_loader import load_dataset, quick_summary
df = load_dataset()
quick_summary(df)
df.head()

## 3. Exploratory Data Analysis

We look at:
- **Class balance** — is the dataset skewed?
- **Feature distributions split by potability** — does any single feature visibly separate safe vs unsafe water?
- **Correlation matrix** — are any features redundant?

In [ ]:
from src.visualize import (
    plot_class_balance, plot_feature_distributions, plot_correlation_heatmap,
)
plot_class_balance(df)
plot_feature_distributions(df)
plot_correlation_heatmap(df)
print('Saved EDA figures to outputs/figures/')

## 4. Preprocessing

Steps:
1. **Stratified train/test split (80/20)** — preserves the potable/non-potable ratio.
2. **Median imputation** — robust to outliers in pH and Sulfate where missing values cluster.
3. **Standardization** — Logistic Regression needs scaled features; tree models tolerate it.

**Important:** the scaler and imputer are fitted on the *training* set only, then applied to the test set. Doing it the other way is data leakage.

In [ ]:
from src.preprocessing import prepare_data
data = prepare_data(df, test_size=0.20, random_state=RANDOM_STATE)
print('Train:', data.X_train.shape, ' Test:', data.X_test.shape)

## 5. Train three classifiers

| Model | Why we picked it |
|---|---|
| **Logistic Regression** | A linear, interpretable baseline. Coefficient signs map directly to "this feature increases / decreases the probability of safe water." |
| **Decision Tree** | Captures non-linear interactions (e.g. *high turbidity AND high TDS*) and is easy to read off as if-then rules — useful for explaining to lab staff. |
| **Random Forest** | An ensemble of decision trees. Reduces overfitting and usually wins on tabular datasets like this one. |

All three use `class_weight='balanced'` to compensate for the class imbalance.

In [ ]:
from src.train_models import train_all, save_models
models = train_all(data)
save_models(models)

## 6. Evaluation

We care about more than accuracy:
- **Recall on the *Non-potable* class** matters most — missing a contaminated sample is a public-health failure.
- **Precision** matters too — false alarms waste limited culture-lab capacity.
- **F1** balances the two.

In [ ]:
from src.evaluate import (
    evaluate_all, best_model, detailed_report, save_metrics, confusion_matrices,
)
from src.visualize import (
    plot_confusion_matrix, plot_model_comparison, plot_feature_importance,
)

metrics = evaluate_all(models, data)
best = best_model(metrics, metric='f1')
_ = detailed_report(best, models[best], data)
save_metrics(metrics, best)
metrics

In [ ]:
for name, cm in confusion_matrices(models, data).items():
    plot_confusion_matrix(cm, name)
plot_model_comparison(metrics)
for name, model in models.items():
    plot_feature_importance(model, data.feature_names, name)
print('Saved performance figures to outputs/figures/')

## 7. Microbiological interpretation

Looking at the Random Forest feature importance, the top drivers of contamination prediction usually are:

- **pH** — values outside 6.5–8.5 favour survival of *Vibrio cholerae* (alkaline) and acid-tolerant enterics; chlorine disinfection efficacy also drops sharply at pH > 8.
- **Chloramines** — secondary disinfectant. Low levels = inadequate disinfection = higher coliform survival.
- **Sulfate / TDS** — high total dissolved solids correlates with surface-water intrusion and sewage contamination.
- **Turbidity** — suspended particles physically *shield* bacteria from UV and chlorine. WHO sets the limit at 5 NTU exactly because efficacy collapses above that.
- **Trihalomethanes (THMs)** — disinfection byproducts. High THMs paradoxically signal that disinfection met heavy organic load, often from upstream contamination.

## 8. Next steps

- Hyperparameter tuning with `GridSearchCV`.
- Add XGBoost / LightGBM.
- SHAP values for per-sample explainability.
- Streamlit dashboard for lab use.
- Cross-reference with **microbial counts** (coliform CFU/100 mL) when available.